In [0]:
from scr.endpoints import list_keys_all, list_keys_changes, items_many, next_offset
from scr.connection import APIClient
from scr.loaders import fetch_all_keys_to_bronze, fetch_items_to_silver_json
import requests
import time
import json

BASE = "https://productapi-synkka.gs1.fi"
EMAIL = dbutils.secrets.get("gs1-kv", "email")
PASSWORD = dbutils.secrets.get("gs1-kv", "password")
GLN = dbutils.secrets.get("gs1-kv", "gln")

account   = "gs1datalake"      # storage accountin nimi
container = "datalake"         # containerin nimi
access_key = "5kEehpTCdCfzcmwOzmq+w4iaiFeMGnfV2OCaxQlGut2kb65ItOD4QDWapkmcT/NI4t8sLaOFAbjL+AStaIorWg=="

# kerrotaan Sparkin asetuksissa, miten mennään storageen
spark.conf.set(f"fs.azure.account.key.{account}.dfs.core.windows.net", access_key)

# muodostetaan polku, johon kirjoitetaan/lukemaan
DELTA_KEYS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/bronze/public_item_sync"
DELTA_PRODUCTS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/silver/catalogue_items"

KEY_BATCH_SIZE = 90000
SINCE_ISO = "2025-09-10T07:03:26.1655687Z"

client = APIClient(BASE, EMAIL, PASSWORD, GLN)
client.authenticate(verbose=True)   

all_keys = fetch_all_keys_to_bronze(spark, client, DELTA_KEYS, batch_size=KEY_BATCH_SIZE)
print(f"Avaimien (All) nouto onnistui, avaimia talletettu: {all_keys}")

time.sleep(1.2)

# Changes → ensin kerrotaan mistä alkaen haetaan
print(f"Aloitetaan avaimien (Changes) nouto alkaen: {SINCE_ISO.split("T", 1)[0]}")
resp_chg = list_keys_changes(client, batch_size=KEY_BATCH_SIZE, since=SINCE_ISO)
chg_items = resp_chg.get("Items") or []
print(f"Avaimien (Changes) nouto onnistui, avaimia haettu: {len(chg_items)}")

time.sleep(1.2)

# muutoksista saadut Id:t
chg_ids = [it.get("Id") for it in chg_items if it.get("Id")]

# haetaan niiden tuotedatat ja tallennetaan Silveriin
n_fetched = fetch_items_to_silver_json(
    spark,
    client,
    silver_path=DELTA_PRODUCTS,
    ids=chg_ids,
    batch_size=1000
)

print(f"Muuttuneiden tuotteiden tiedot noudettu ja tallennettu Silveriin. Rivimäärä: {n_fetched}")